In [1]:
# =====================================================================
# CELLA IMPORTAZIONI LIBRERIE PER MAC
# =====================================================================
import os
import glob
import logging
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from keras.models import load_model

# Silenziamo i log inutili del Mac
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Blocca tutto tranne gli errori fatali
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0' 
logging.getLogger('tensorflow').setLevel(logging.ERROR)

# NOTA: Rimosso 'TF_CUDNN_USE_AUTOTUNE' perché sul tuo Mac non serve

import tensorflow as tf

# Altri silenziatori di log 
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

# =====================================================================
# MODIFICA 1: DISABILITARE LA GPU PER EVITARE I GRADIENTI NaN (ESPLOSIONE DELLA LOSS)
# =====================================================================
# Diciamo a TensorFlow di "nascondere" la GPU M1 (gestita da tensorflow-metal).
tf.config.set_visible_devices([], 'GPU')

# Riga di controllo per essere sicuri al 100% che abbia funzionato
print("Dispositivi di calcolo attivi:", tf.config.get_visible_devices())
# =====================================================================

# Import di Keras (lasciati identici a quelli del tuo collega)
from keras import layers, models, losses
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

Dispositivi di calcolo attivi: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [2]:
# ==============================================================================
# DATA ENGINE V9 (Caricamento Globale in RAM)
# ==============================================================================
def load_and_process_all_files(file_list, alpha=0.20):
    X_all, Y_all = [], []
    print(f"Inizio caricamento ed EMA Decluttering di {len(file_list)} file...")
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   
        people_xy = data['people_xy']   
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
        mag_reshaped = mag.reshape(T, 1, 120, 18) 
        
        # Inizializza il background per l'EMA
        bg = np.copy(mag_reshaped[0])
        decluttered = np.zeros_like(mag_reshaped)
        
        # L'EMA viene calcolato qui, una volta per tutte, in perfetto ordine cronologico!
        for t in range(T):
            bg = alpha * mag_reshaped[t] + (1 - alpha) * bg
            decluttered[t] = np.abs(mag_reshaped[t] - bg)
        
        # Flatten delle coordinate e concatenazione con la mask (12 valori totali)
        flat_coords = people_xy.reshape(T, 8)
        combined_target = np.concatenate([flat_coords, people_mask], axis=1)

        X_all.append(decluttered)
        Y_all.append(combined_target)
        
        print(f"File {i+1}/{len(file_list)} processato. ({T} frame pre-calcolati)")

    # Uniamo tutte le liste in due immensi tensori Numpy pronti per la GPU
    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)
    return X, Y

# ==============================================================================
# 2. SPLIT DIVERSI SEGUENDO DIVERSI CRITERI
# ==============================================================================
# Split1 (78% train e 22% val), > windows con 3/4 persone in train
val_indices = [23, 20, 3, 15, 7, 11] 
train_indices = [22, 16, 17, 18, 19, 21, 0, 1, 2, 4, 12, 13, 14, 5, 6, 8, 9, 10]
# Split2 (80% train e 22% val), stesso cocetto di split 1 "STRESS TEST sul MULTIPATH"
val_indices = [23, 20, 0, 13, 9] 
train_indices = [22, 16, 17, 18, 19, 21, 1, 2, 3, 4, 12, 14, 15, 5, 6, 7, 8, 10, 11]
# Split3 (78% train e 22% val) > equilibrato tra train e val 
#val_indices = [22, 20, 2, 12, 14, 6] 
#train_indices = [23, 16, 18, 19, 21, 0, 1, 3, 4, 13, 15, 5, 7, 8, 9, 10, 11, 17]

#tutti_i_file = glob.glob("dataset/data/*.npz")
tutti_i_file = glob.glob("dataset/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

# ==============================================================================
# 3. ESECUZIONE DEL MOTORE (ATTENZIONE: Ci metterà ~1 minuto a caricare tutto in RAM)
# ==============================================================================
print("\n--- PREPARAZIONE TRAINING SET ---")
X_train, Y_train = load_and_process_all_files(train_files)

print("\n--- PREPARAZIONE VALIDATION SET ---")
X_val, Y_val = load_and_process_all_files(val_files)

print("\n==================================================")
print(f"DATI TOTALI PRONTI IN RAM!")
print(f"Totale FRAME individuali di Train:      {X_train.shape[0]}")
print(f"Totale FRAME individuali di Validation: {X_val.shape[0]}")
print("==================================================")


--- PREPARAZIONE TRAINING SET ---
Inizio caricamento ed EMA Decluttering di 19 file...
File 1/19 processato. (7500 frame pre-calcolati)
File 2/19 processato. (7500 frame pre-calcolati)
File 3/19 processato. (7500 frame pre-calcolati)
File 4/19 processato. (7500 frame pre-calcolati)
File 5/19 processato. (7500 frame pre-calcolati)
File 6/19 processato. (7500 frame pre-calcolati)
File 7/19 processato. (7500 frame pre-calcolati)
File 8/19 processato. (7500 frame pre-calcolati)
File 9/19 processato. (7500 frame pre-calcolati)
File 10/19 processato. (7500 frame pre-calcolati)
File 11/19 processato. (7500 frame pre-calcolati)
File 12/19 processato. (7500 frame pre-calcolati)
File 13/19 processato. (7500 frame pre-calcolati)
File 14/19 processato. (7500 frame pre-calcolati)
File 15/19 processato. (7500 frame pre-calcolati)
File 16/19 processato. (7500 frame pre-calcolati)
File 17/19 processato. (7500 frame pre-calcolati)
File 18/19 processato. (7500 frame pre-calcolati)
File 19/19 processato

In [3]:
# =====================================================================
# BULGARIAN SQUAT PER MAC
# =====================================================================
import itertools
PERM_INDICES = tf.constant(list(itertools.permutations([0, 1, 2, 3])), dtype=tf.int32)

def hungarian_total_loss(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1) 
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)

    y_true_coords_exp = tf.expand_dims(y_true_coords, 1) 
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3]) 
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)

    #bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3]) 

    total_cost = coords_cost_norm + (1.5 * mask_cost_norm) 
    return tf.reduce_min(total_cost, axis=1) 

def hungarian_rmse_metres(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    
    min_coords_cost = tf.reduce_min(coords_cost, axis=1)
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    
    return tf.sqrt(min_coords_cost / num_valid_people)

def hungarian_mask_acc(y_true, y_pred):
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)
    
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)
    
    #bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3])
    
    total_cost = coords_cost_norm + (1.5 * mask_cost_norm)
    best_perm_idx = tf.argmin(total_cost, axis=1, output_type=tf.int32)
    
    batch_size = tf.shape(y_pred)[0]
    gather_nd_indices = tf.stack([tf.range(batch_size, dtype=tf.int32), best_perm_idx], axis=1)
    best_mask_pred = tf.gather_nd(y_pred_mask_perm, gather_nd_indices)
    
    return tf.reduce_mean(tf.keras.metrics.binary_accuracy(y_true_mask, best_mask_pred))

In [4]:
# ==============================================================================
# ARCHITETTURA EEAI-NET V2 - Squeeze-and-Excitation (SE)
# ==============================================================================

def se_block(input_tensor, reduction=8, name_prefix="se"):
    """
    Squeeze-and-Excitation block ottimizzato per serie temporali 1D/2D.
    Calcola l'importanza di ciascuna feature map e la ricalibra.
    """
    # Recupera il numero di canali (es. 32, 64, 128)
    channels = int(input_tensor.shape[-1])
    
    # Squeeze: Global Average Pooling (riduce a un vettore di lunghezza 'channels')
    se = layers.GlobalAveragePooling2D(name=f"{name_prefix}_gap")(input_tensor)
    
    # Excitation: Due layer fully connected con bottleneck
    se = layers.Dense(channels // reduction, activation='relu', use_bias=False, name=f"{name_prefix}_dense1")(se)
    se = layers.Dense(channels, activation='sigmoid', use_bias=False, name=f"{name_prefix}_dense2")(se)
    
    # Reshape per permettere il broadcasting durante la moltiplicazione (1, 1, C)
    se = layers.Reshape((1, 1, channels), name=f"{name_prefix}_reshape")(se)
    
    # Ricalibrazione: Moltiplica l'input originale per i pesi di attenzione
    return layers.Multiply(name=f"{name_prefix}_mul")([input_tensor, se])

def build_eeai_model_v2_se(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")

    # BLOCCO 1
    x = layers.Conv2D(32, (1, 9), padding='same', activation='relu', name="conv_1a")(inputs)
    x = layers.MaxPooling2D((1, 2), name="pool_1")(x)
    x = layers.Conv2D(32, (1, 9), padding='same', activation='relu', name="conv_1b")(x)
    x = se_block(x, reduction=8, name_prefix="se_blk1") # <-- INSERIMENTO SE 1
    x = layers.MaxPooling2D((1, 2), name="pool_2")(x) 
    
    # BLOCCO 2
    x = layers.SeparableConv2D(64, (1, 3), padding='same', activation='relu', name="sep_conv_1")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_3")(x)
    x = layers.SeparableConv2D(64, (1, 3), padding='same', activation='relu', name="sep_conv_2")(x)
    x = se_block(x, reduction=8, name_prefix="se_blk2") # <-- INSERIMENTO SE 2
    x = layers.MaxPooling2D((1, 2), name="pool_4")(x) 

    # BLOCCO 3
    x = layers.Conv2D(128, (1, 3), padding='same', activation='relu', name="conv_3a")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_5")(x)
    x = layers.Conv2D(128, (1, 3), padding='same', activation='relu', name="conv_3b")(x)
    x = se_block(x, reduction=8, name_prefix="se_blk3") # <-- INSERIMENTO SE 3
    x = layers.MaxPooling2D((1, 2), name="pool_6")(x)
    
    x = layers.Conv2D(128, (1, 3), padding='same', activation='relu', name="conv_3c")(x)
    
    # Flatten e Head Densly Connected
    x = layers.Flatten(name="flatten_features")(x)
    
    x = layers.Dense(256, activation='relu', name="features_deep2")(x)
    x = layers.Dropout(0.2, name="drop_features")(x) 
    common_feat = layers.Dense(128, activation='relu', name="features")(x)

    # --- OUTPUT MULTI-HEAD INVARIATI ---
    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)
    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])

    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V2_SE")

In [5]:
# ==============================================================================
# ADDESTRAMENTO MODELLO V2 (Con Squeeze-and-Excitation)
# ==============================================================================

# Inizializzazione del nuovo modello con SE
model_se = build_eeai_model_v2_se()

# Compilazione 
model_se.compile(
    optimizer='adam',
    loss=hungarian_total_loss, 
    metrics=[hungarian_rmse_metres, hungarian_mask_acc]
)

# Cambiato il nome del salvataggio per non sovrascrivere il vecchio modello
checkpoint_se = ModelCheckpoint("eeai_best_model_romano_v2_se.keras", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)

EPOCHS = 50 

print("\n--- INIZIO ADDESTRAMENTO OTTIMIZZATO (MODELLO SE) ---")
history_se = model_se.fit(
    X_train, Y_train,                
    validation_data=(X_val, Y_val),  
    batch_size=32,                   
    shuffle=True,                    
    epochs=EPOCHS,
    callbacks=[checkpoint_se, reduce_lr, early_stop], 
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")


--- INIZIO ADDESTRAMENTO OTTIMIZZATO (MODELLO SE) ---
Epoch 1/50
4454/4454 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - hungarian_mask_acc: 0.8406 - hungarian_rmse_metres: 0.8044 - loss: 1.5723
Epoch 1: val_loss improved from None to 0.65431, saving model to eeai_best_model_romano_v2_se.keras

Epoch 1: finished saving model to eeai_best_model_romano_v2_se.keras
4454/4454 ━━━━━━━━━━━━━━━━━━━━ 43s 9ms/step - hungarian_mask_acc: 0.9117 - hungarian_rmse_metres: 0.6368 - loss: 0.9134 - val_hungarian_mask_acc: 0.9438 - val_hungarian_rmse_metres: 0.4850 - val_loss: 0.6543 - learning_rate: 0.0010
Epoch 2/50
4448/4454 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - hungarian_mask_acc: 0.9586 - hungarian_rmse_metres: 0.4969 - loss: 0.5190
Epoch 2: val_loss improved from 0.65431 to 0.58031, saving model to eeai_best_model_romano_v2_se.keras

Epoch 2: finished saving model to eeai_best_model_romano_v2_se.keras
4454/4454 ━━━━━━━━━━━━━━━━━━━━ 40s 9ms/step - hungarian_mask_acc: 0.9599 - hungarian_rmse_metres: 0.4789 - los

In [6]:
# ==============================================================================
# MODEL SUMMARY PER EMBEDDED
# ==============================================================================

def embedded_summary(model, input_shape=(1, 120, 18)):
    
    # 2. Calcola i parametri statici (Flash)
    total_params = model.count_params()
    estimated_flash_kb = (total_params * 4) / 1024
    
   # 3. Calcola il picco di memoria dinamica (SRAM/Tensor Arena)
    max_layer_ram_kb = 0
    for layer in model.layers:
        # AGGIUNTO: Salta l'InputLayer o i layer senza output_shape per evitare l'AttributeError
        if layer.__class__.__name__ == 'InputLayer' or not hasattr(layer, 'output_shape'):
            continue
            
        output_shape = layer.output_shape
        if isinstance(output_shape, list):
            num_elements = sum([np.prod([dim for dim in shape[1:] if dim is not None]) for shape in output_shape])
        else:
            num_elements = np.prod([dim for dim in output_shape[1:] if dim is not None])
            
        layer_ram_kb = (num_elements * 4) / 1024
        if layer_ram_kb > max_layer_ram_kb:
            max_layer_ram_kb = layer_ram_kb

    input_elements = np.prod(input_shape)
    input_ram_kb = (input_elements * 4) / 1024
    peak_arena_kb = input_ram_kb + max_layer_ram_kb

    # 4. Stampa il verdetto 
    print("============================================")
    print("   REPORT REQUISITI HARDWARE (STIMA FLOAT32)   ")
    print("============================================")
    print(f" Memoria FLASH stimata : {estimated_flash_kb:.2f} KB  (Limite : < 800 KB)")
    print(f" Memoria SRAM stimata  : ~{peak_arena_kb:.2f} KB (Limite : < 400 KB)")
    print(" Operazioni Ricorrenti : ASSENTI (RNN/LSTM/GRU non rilevate)")
    print(" Nota sulla Quantizz.  : Raccomandata INT8 per ESP32-S3 (ridurrà la RAM di ~4x)")
    print("============================================\n")

embedded_summary(model_se)
model_se.summary()

   REPORT REQUISITI HARDWARE (STIMA FLOAT32)   
 Memoria FLASH stimata : 848.17 KB  (Limite : < 800 KB)
 Memoria SRAM stimata  : ~8.44 KB (Limite : < 400 KB)
 Operazioni Ricorrenti : ASSENTI (RNN/LSTM/GRU non rilevate)
 Nota sulla Quantizz.  : Raccomandata INT8 per ESP32-S3 (ridurrà la RAM di ~4x)



Model: "EEAI_Net_V2_SE"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ radar_input         │ (None, 1, 120,    │          0 │ -                 │
│ (InputLayer)        │ 18)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_1a (Conv2D)    │ (None, 1, 120,    │      5,216 │ radar_input[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_1              │ (None, 1, 60, 32) │          0 │ conv_1a[0][0]     │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_1b (Conv2D)    │ (None, 1, 60, 32) │      9,248 │ pool_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ se_blk1_gap         │ (None, 32)        │          0 │ conv_1b[0][0]     │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ se_blk1_dense1      │ (None, 4)         │        128 │ se_blk1_gap[0][0] │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ se_blk1_dense2      │ (None, 32)        │        128 │ se_blk1_dense1[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ se_blk1_reshape     │ (None, 1, 1, 32)  │          0 │ se_blk1_dense2[0… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ se_blk1_mul         │ (None, 1, 60, 32) │          0 │ conv_1b[0][0],    │
│ (Multiply)          │                   │            │ se_blk1_reshape[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_2              │ (None, 1, 30, 32) │          0 │ se_blk1_mul[0][0] │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sep_conv_1          │ (None, 1, 30, 64) │      2,208 │ pool_2[0][0]      │
│ (SeparableConv2D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_3              │ (None, 1, 15, 64) │          0 │ sep_conv_1[0][0]  │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sep_conv_2          │ (None, 1, 15, 64) │      4,352 │ pool_3[0][0]      │
│ (SeparableConv2D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ se_blk2_gap         │ (None, 64)        │          0 │ sep_conv_2[0][0]  │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ se_blk2_dense1      │ (None, 8)         │        512 │ se_blk2_gap[0][0] │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ se_blk2_dense2      │ (None, 64)        │        512 │ se_blk2_dense1[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ se_blk2_reshape     │ (None, 1, 1, 64)  │          0 │ se_blk2_dense2[0… │
│ (Reshape)           │                   │            │                 

 Total params: 651,398 (2.48 MB)

 Trainable params: 217,132 (848.17 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 434,266 (1.66 MB)

In [9]:
# ==============================================================================
# VISUALIZZATORE 3.1 (Aggiornato per V2 SE)
# ==============================================================================

file_target = "dataset/window_000000.npz"
if not os.path.exists(file_target):
    print(f"ERRORE: Non trovo il file {file_target}")
else:
    data = np.load(file_target)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    print("Elaborazione filtri e previsioni in corso (V9 con SE)...")
    mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2).reshape(T, 1, 120, 18)
    decluttered = np.zeros_like(mag)
    bg = np.copy(mag[0])
    alpha = 0.20
    for t in range(T):
        bg = alpha * mag[t] + (1 - alpha) * bg
        decluttered[t] = np.abs(mag[t] - bg)

    print("Caricamento dei pesi migliori dal file .keras (Modello SE) ...")
    
    # Caricamento del NUOVO modello 
    model_loaded = load_model(
        "eeai_best_model_romano_v2_se.keras",
        custom_objects={
            "hungarian_total_loss": hungarian_total_loss,
            "hungarian_rmse_metres": hungarian_rmse_metres, 
            "hungarian_mask_acc": hungarian_mask_acc
        }
    )

    preds = model_loaded.predict(decluttered, verbose=0)
    
    p_coords = preds[:, :8].reshape(T, 4, 2)
    p_mask = preds[:, 8:]
    
    print("Dati pronti! Inizializzazione Radar...")

    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(4.8, 7.2))
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            
            ax.set_title(f"Radar V9 (SE) | Frame: {frame_idx}/{T-1} | Window: 5", fontsize=14, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            for i in range(4):
                is_present = bool(gt_mask[frame_idx, i] > 0.5)
                if is_present:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=120, edgecolors='black', marker='o', label='REALE (GT)' if i==0 else "")
                    ax.text(rx, ry + 0.2, f"P{i+1}", color='darkgreen', fontweight='bold', ha='center')

                conf = float(p_mask[frame_idx, i])
                if conf >= soglia:
                    px, py = p_coords[frame_idx, i]
                    alpha_val = max(0.3, conf)
                    ax.scatter(px, py, c='red', s=100, marker='X', edgecolors='darkred', alpha=alpha_val, label='PREDETTO' if i==0 else "")
                    ax.text(px, py - 0.3, f"{conf*100:.0f}%", color='red', fontsize=10, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:')
    slider_soglia = widgets.FloatSlider(value=0.50, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    controls = widgets.VBox([slider_frame, slider_soglia])
    controls.layout.margin = '20px 20px 20px 0px' 
    ui = widgets.HBox([controls, out])
    
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)

Elaborazione filtri e previsioni in corso (V9 con SE)...
Caricamento dei pesi migliori dal file .keras (Modello SE) ...
Dati pronti! Inizializzazione Radar...
